In [1]:
import json, shutil, os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits
threadpool_limits(limits=1)

In [2]:
TASK = Path.cwd().parent
INPUT, OUTPUT = TASK / 'input', TASK / 'output'

In [3]:
config = json.loads((INPUT / 'config.json').read_text())
SEED = config['seed']
rng = np.random.default_rng(SEED)

In [4]:
for cell in json.loads((INPUT / 'helpers.ipynb').read_text())['cells']:
    exec(''.join(cell['source']), globals())
carry(INPUT, OUTPUT)

In [5]:
queries = pd.read_csv(INPUT / 'queries.csv').query("split == 'test'")
blocks = np.load(INPUT / 'query_blocks.npz')['blocks']
reference_data = np.load(INPUT / 'reference_panels.npz')
fits = pd.read_csv(INPUT / 'fitted_parameters.csv')
marker_order = np.load(INPUT / 'marker_order.npy')

In [6]:
# blind inference: test truth is first read in f_evaluate
prediction_rows, predictions, probabilities, unknown_scores, rejected = [], [], [], [], []
for fit in fits.to_dict('records'):
    panel, markers, steps = fit['panel'], int(fit['markers']), int(fit['steps'])
    references = reference_data[f'test_{panel}']
    for query in queries.query('steps == @steps').to_dict('records'):
        cost = copy_cost(blocks[query['query_id']], references, marker_order[:markers], reference_data['scale'])
        references_per_source = references.shape[1]
        local = cost.argmin(axis=1) // references_per_source
        global_path = np.full(config['blocks'], cost.sum(axis=0).argmin() // references_per_source)
        copying = viterbi(cost, fit['lambda']) // references_per_source
        local_p = source_probabilities(softmax(-cost / fit['temperature'], axis=1))
        copying_p = source_probabilities(posterior(cost, fit['lambda'], fit['temperature']))
        global_p = np.tile(source_probabilities(softmax(-cost.sum(axis=0) / fit['temperature'])), (config['blocks'], 1))
        for method, path, p in [('global', global_path, global_p), ('nearest', local, local_p), ('copying', copying, copying_p)]:
            prediction_rows.append({'prediction_id': len(prediction_rows), 'panel': panel, 'markers': markers,
                                    'method': method, **query})
            predictions.append(path)
            probabilities.append(p)
            unknown_scores.append(cost.min(axis=1))
            rejected.append(cost.min(axis=1) > fit['rejection_threshold'])
np.savez_compressed(OUTPUT / 'predictions.npz', paths=np.stack(predictions), probabilities=np.stack(probabilities),
                    unknown_score=np.stack(unknown_scores), rejected=np.stack(rejected))
pd.DataFrame(prediction_rows).to_csv(OUTPUT / 'prediction_index.csv', index=False)
pd.DataFrame(prediction_rows).groupby(['panel', 'markers', 'method']).size().rename('predictions').to_frame()

predictions
panel markers method              
exact 1       copying          248
              global           248
              nearest          248
      4       copying          248
              global           248
              nearest          248
      16      copying          248
              global           248
              nearest          248
      64      copying          248
              global           248
              nearest          248
      600     copying          248
              global           248
              nearest          248
proxy 1       copying          248
              global           248
              nearest          248
      4       copying          248
              global           248
              nearest          248
      16      copying          248
              global           248
              nearest          248
      64      copying          248
              global           248
              nearest          248
      600     copying          248
              global           248
              nearest          248